# Note 02: Threads, Blocks, and Grids

A GPU kernel runs many copies of the same code. This note explains how CUDA organizes those copies into threads, blocks, and grids, then builds a global thread ID for assigning work.

## The CUDA execution hierarchy

A one-dimensional kernel launch has this form:

```cpp
kernel<<<number_of_blocks, threads_per_block>>>();
```

For example, `kernel<<<3, 4>>>()` launches three blocks with four threads each, for twelve threads in total. CUDA groups threads as:

```text
Grid
  blocks
    threads
```

## Inspecting block and thread IDs

`threadIdx.x` identifies a thread inside its block. `blockIdx.x` identifies the block. The order of lines printed by the GPU is not guaranteed because threads execute concurrently.

In [ ]:
%%writefile threads_blocks.cu
#include <cuda_runtime.h>

__global__ void show_threads() {
    printf("Block %d, Thread %d\n",
           blockIdx.x, threadIdx.x);
}

int main() {
    show_threads<<<3, 4>>>();
    cudaDeviceSynchronize();
    return 0;
}

In [ ]:
!nvcc threads_blocks.cu -o threads_blocks
!./threads_blocks

## The global thread ID

`threadIdx.x` repeats in every block, so it is not enough to identify a position in the full grid. The standard one-dimensional global index is:

```cpp
int i = blockIdx.x * blockDim.x + threadIdx.x;
```

For `<<<3, 4>>>`, this produces IDs from `0` through `11`. `blockDim.x` is the number of threads in each block, while `gridDim.x` is the total number of blocks.

In [ ]:
%%writefile global_id.cu
#include <cuda_runtime.h>

__global__ void show_global_id() {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    printf("Block: %d | Thread: %d | Global ID: %d\n",
           blockIdx.x, threadIdx.x, i);
}

int main() {
    show_global_id<<<3, 4>>>();
    cudaDeviceSynchronize();
    return 0;
}

In [ ]:
!nvcc global_id.cu -o global_id
!./global_id

## What this note establishes

- a launch config defines blocks and threads per block
- `threadIdx.x` is local to a block
- `blockIdx.x` identifies the current block
- `blockDim.x` and `gridDim.x` describe the launch shape
- global indexing maps each thread to one position in a larger problem